# Week 46, Day 4: Attention as normalisation

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w046/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Two regimes of one model

Complete model(): compute the attended drive A E, the suppressive drive by convolving it with the normalised pooling kernel, and the response of the central neuron. Then compute contrast-response functions with and without attention in both configurations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
x = np.linspace(-10, 10, 201)
k = np.exp(-x ** 2 / (2 * 5.0 ** 2)); k /= k.sum()

def model(c, stim_w, att_w, attend=True, sigma=0.05, att_gain=3.0):
    E = c * np.exp(-x ** 2 / (2 * stim_w ** 2))
    A = 1 + (att_gain - 1) * np.exp(-x ** 2 / (2 * att_w ** 2)) if attend else np.ones_like(x)
    ...
    return R[100]                                        # the neuron at the centre

cs = np.logspace(-3, 0, 31)
c50 = lambda R: np.interp(R.max() / 2, R, cs)
c50_ratio, rmax_ratio = {}, {}
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
for a_, (name, sw, aw) in zip(ax, [("small stimulus", 1.0, 6.0), ("large stimulus", 6.0, 1.0)]):
    un = np.array([model(c, sw, aw, False) for c in cs]); at = np.array([model(c, sw, aw, True) for c in cs])
    c50_ratio[name], rmax_ratio[name] = c50(at) / c50(un), at.max() / un.max()
    a_.semilogx(cs, un, label="unattended"); a_.semilogx(cs, at, label="attended"); a_.set_title(name); a_.set_xlabel("contrast"); a_.legend()
    print(f"{name}: c50 ratio {c50_ratio[name]:.2f}, Rmax ratio {rmax_ratio[name]:.2f}")
plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
def test_contrast_gain():
    "small stimulus, large attention field: c50 at least halves while Rmax rises by under 25%"
    assert c50_ratio["small stimulus"] < 0.6 and rmax_ratio["small stimulus"] < 1.25
def test_response_gain():
    "large stimulus, small attention field: Rmax rises by more than 60%"
    assert rmax_ratio["large stimulus"] > 1.6

test_contrast_gain()
test_response_gain()
print("All checks passed")

### Exercise: Attention biases the competition

Write pair_response(c1, c2, a1=1, a2=1) from the formula with w1 = 60, w2 = 10, sigma = 0.05, and evaluate the five conditions at contrast 0.5 with an attention gain of 4.

In [ ]:
def pair_response(c1, c2, a1=1.0, a2=1.0, w1=60.0, w2=10.0, sigma=0.05):
    return ...

resp = {
    "preferred alone": pair_response(0.5, 0.0),
    "non-preferred alone": pair_response(0.0, 0.5),
    "pair": pair_response(0.5, 0.5),
    "attend preferred": pair_response(0.5, 0.5, a1=4.0),
    "attend non-preferred": pair_response(0.5, 0.5, a2=4.0),
}
for k_, v in resp.items():
    print(f"{k_:22s} {v:5.1f} spikes/s")

def softmax_attention(q, K, V):
    s = K @ q; w = np.exp(s - s.max()); w /= w.sum()      # divisive normalisation of the weights
    return w @ V, w
_, w = softmax_attention(np.array([2.0, 0.0]), np.array([[1.0, 0.0], [0.0, 1.0]]), np.array([[60.0], [10.0]]))
print("softmax weights when the query matches the first key:", np.round(w, 2))

In [ ]:
import numpy as np
def test_pair():
    "without attention, the pair response lies between the single responses"
    assert resp["non-preferred alone"] < resp["pair"] < resp["preferred alone"]
def test_bias():
    "attending to the preferred stimulus moves the pair response at least two thirds of the way to its single response"
    assert (resp["attend preferred"] - resp["pair"]) > (2 / 3) * (resp["preferred alone"] - resp["pair"])
def test_bias_other():
    "attending to the non-preferred stimulus moves it towards the non-preferred response"
    assert resp["attend non-preferred"] < resp["pair"] - 10

test_pair()
test_bias()
test_bias_other()
print("All checks passed")